# 06 — RoBERTa (AutoModelForMultipleChoice) for the Smart MCQ Solver

## 1. Setup

Imports + reproducibility seeds + device detection.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup

In [2]:
import random
# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

In [3]:
# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


## 2. Configuration

Key choices i made here:
- **`max_len=128`** — per-option length (prompt + ONE option), not all 5 at once
- **`batch_size=8`** — each example is 5x wider than DeBERTa's per-option batches (5 choices stacked)
- **`lr=2e-5`** — standard transformer fine-tuning LR
- **`epochs=5`** — transformers converge fast on small datasets
- **`warmup_ratio=0.1`** — 10% of steps ramp LR from 0 → max before decaying

In [ ]:
CFG = dict(
    model_name   = 'roberta-base',
    max_len      = 128,     # per-option length; prompt + one option, not all 5 at once
    batch_size   = 8,       # 5x wider per example than DeBERTa's per-option batches (5 choices stacked)
    lr           = 2e-5,
    weight_decay = 0.01,
    epochs       = 5,
    warmup_ratio = 0.1,
    patience     = 3,
    seed         = 42,
)

set_seed(CFG["seed"])

In [5]:
DATA_DIR = Path('/kaggle/input/competitions/smart-mcq-solver-challenge')
if not DATA_DIR.exists():
    DATA_DIR = Path('data')

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
MODEL_DIR  = OUTPUT_DIR / 'models'
PRED_DIR   = OUTPUT_DIR / 'predictions'
for d in (OUTPUT_DIR, MODEL_DIR, PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [6]:
# Answer <-> int maps
ANSWER_MAP  = {'A':0,'B':1,'C':2,'D':3,'E':4}
REVERSE_MAP = {v:k for k,v in ANSWER_MAP.items()}
OPTION_COLS = list('ABCDE')
print(CFG)

{'model_name': '/kaggle/input/models/sachin62/roberta-base/transformers/default/1', 'max_len': 128, 'batch_size': 8, 'lr': 2e-05, 'weight_decay': 0.01, 'epochs': 5, 'warmup_ratio': 0.1, 'patience': 3, 'seed': 42}


## 3. Load data + stratified split

In [7]:
def load_raw():
    """Load competition CSVs."""
    train_df = pd.read_csv(DATA_DIR / 'train.csv')
    test_df  = pd.read_csv(DATA_DIR / 'test.csv')
    return train_df, test_df

In [8]:
def lowercase_columns(df, cols=['prompt']+OPTION_COLS):
    """Lowercase + strip whitespace on every text column."""
    df = df.copy()
    for col in cols:
        df[col] = df[col].astype(str).str.lower().str.strip()
    return df

In [9]:
def stratified_split(train_df, val_size=0.2, seed=CFG['seed']):
    """Manual stratified split on the 'answer' column."""
    np.random.seed(seed)
    train_idx, val_idx = [], []
    for ans in 'ABCDE':
        idx = train_df[train_df['answer']==ans].index.tolist()
        np.random.shuffle(idx)
        cut = int(len(idx)*(1.0-val_size))
        train_idx += idx[:cut]
        val_idx   += idx[cut:]
    tr = train_df.loc[train_idx].reset_index(drop=True)
    va = train_df.loc[val_idx].reset_index(drop=True)
    return tr, va

In [10]:
# Load + clean
train_raw, test_df = load_raw()
train_raw = lowercase_columns(train_raw)
test_df   = lowercase_columns(test_df)

# Stratified 80/20
tr_df, va_df = stratified_split(train_raw)
print(f'train: {len(tr_df)} | val: {len(va_df)} | test: {len(test_df)}')

train: 1599 | val: 401 | test: 500


## 4. Dataset — the multiple-choice input format

`AutoModelForMultipleChoice` expects, per example, a **stack of 5 encodings**
(one per option) each shaped `(seq_len,)`, giving a batch tensor of shape
`(batch, 5, seq_len)`.

Each of the 5 encodings is `(prompt, option_text)` tokenized as a pair —
the model sees the full question once per option, and produces one logit
per option, then a joint softmax over the 5.

In [11]:
class MCQDatasetHF(Dataset):
    """Dataset for AutoModelForMultipleChoice — returns 5 stacked encodings per example."""
    def __init__(self, df, tokenizer, max_len, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt  = row['prompt']
        options = [row[c] for c in OPTION_COLS]

        # Tokenize (prompt, option) as a pair, once per option
        enc = self.tok(
            [prompt]*5, options,
            max_length=self.max_len, truncation=True, padding='max_length',
            return_tensors='pt',
        )
        item = {
            'input_ids':      enc['input_ids'],       # (5, max_len)
            'attention_mask': enc['attention_mask'],   # (5, max_len)
        }
        if not self.is_test:
            item['labels'] = torch.tensor(ANSWER_MAP[row['answer']], dtype=torch.long)
        return item

In [12]:
# Load tokenizer + build datasets/loaders
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])

tr_ds = MCQDatasetHF(tr_df, tokenizer, CFG['max_len'], is_test=False)
va_ds = MCQDatasetHF(va_df, tokenizer, CFG['max_len'], is_test=False)
te_ds = MCQDatasetHF(test_df, tokenizer, CFG['max_len'], is_test=True)

tr_loader = DataLoader(tr_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=0, pin_memory=True)
va_loader = DataLoader(va_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=0, pin_memory=True)
te_loader = DataLoader(te_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=0, pin_memory=True)

In [13]:
# Quick check: pull one batch and inspect shapes
batch = next(iter(tr_loader))
print('input_ids shape :', batch['input_ids'].shape)   # (batch, 5, max_len)
print('labels shape    :', batch['labels'].shape)

input_ids shape : torch.Size([8, 5, 128])
labels shape    : torch.Size([8])


## 5. Model

`AutoModelForMultipleChoice` already includes the multiple-choice
classification head (a linear layer producing one logit per choice from the
pooled `[CLS]` representation) and computes cross-entropy loss internally
when `labels` are passed — no need to write a custom head or loss, unlike
the DeBERTa notebook's custom `DeBERTaOptionScorer`.


In [14]:
# Load pretrained RoBERTa with the multiple-choice head
model = AutoModelForMultipleChoice.from_pretrained(CFG['model_name']).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'{CFG["model_name"]} loaded — {total_params/1e6:.1f}M params')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: /kaggle/input/models/sachin62/roberta-base/transformers/default/1
Key                         | Status     | 
----------------------------+------------+-
classifier.dense.weight     | UNEXPECTED | 
classifier.dense.bias       | UNEXPECTED | 
classifier.out_proj.weight  | UNEXPECTED | 
classifier.out_proj.bias    | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 
roberta.pooler.dense.weight | MISSING    | 
roberta.pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


/kaggle/input/models/sachin62/roberta-base/transformers/default/1 loaded — 124.6M params


## 6. MAP@3 metric

Same MAP@3 formula used across the project:

| Position of correct answer | Score |
|----------------------------|-------|
| 1st                        | 1.00  |
| 2nd                        | 0.50  |
| 3rd                        | 0.33  |
| Not in top-3               | 0.00  |

In [15]:
def map_at_3_from_logits(logits, labels):
    """Compute MAP@3 from raw logits and integer labels."""
    probs  = torch.softmax(logits, dim=1).cpu().numpy()
    labels = labels.cpu().numpy()
    scores = []
    for i, true in enumerate(labels):
        top3 = np.argsort(probs[i])[-3:][::-1]
        hit  = np.where(top3 == true)[0]
        scores.append(1.0/(hit[0]+1) if len(hit) else 0.0)
    return float(np.mean(scores))

## 7. WandB setup

In [16]:
import wandb
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret('WANDB_API_KEY')
    wandb.login(key=key)
    USE_WANDB = True
    print('[OK] WandB logged in!')
except Exception:
    USE_WANDB = False
    print('WandB secret not found — continuing without it.')

if USE_WANDB:
    run = wandb.init(
        project='23f2003236-t22026',
        name='roberta_mc_v1',
        config=CFG,
        tags=['roberta','pretrained','transformer','multiple-choice'],
        reinit=True
    )
    print(f'Run URL: {run.url}')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f2003236 (23f2003236-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[OK] WandB logged in!


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260714_155558-7awxxarm
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run roberta_mc_v1
wandb: ⭐️ View project at https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026
wandb: 🚀 View run at https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026/runs/7awxxarm


Run URL: https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026/runs/7awxxarm


## 8. Training loop

Standard fine-tuning recipe:
- **AdamW + linear warmup/decay** — standard transformer schedule
- **Mixed precision** (`autocast` + `GradScaler`) — ~2x faster on T4
- **Gradient clipping** (`max_norm=1.0`) — prevents exploding gradients
- **Early stopping** on val MAP@3 (patience=3) — saves best checkpoint

In [17]:
# Optimizer + scheduler + scaler
optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
total_steps  = len(tr_loader) * CFG['epochs']
warmup_steps = int(total_steps * CFG['warmup_ratio'])
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))

/tmp/ipykernel_24/3651208761.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))


In [18]:
def train_epoch():
    """One training epoch — mixed precision + grad clip."""
    model.train()
    total_loss = 0.0
    for batch in tr_loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbl  = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
            out  = model(input_ids=ids, attention_mask=mask, labels=lbl)
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(tr_loader)

In [19]:
@torch.no_grad()
def evaluate(loader):
    """Evaluate on a given loader — returns (avg_loss, MAP@3)."""
    model.eval()
    total_loss, all_logits, all_labels = 0.0, [], []
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbl  = batch['labels'].to(DEVICE)
        out  = model(input_ids=ids, attention_mask=mask, labels=lbl)
        total_loss += out.loss.item()
        all_logits.append(out.logits)
        all_labels.append(lbl)
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    map3 = map_at_3_from_logits(all_logits, all_labels)
    return total_loss/len(loader), map3

In [20]:
# Training loop with early stopping
best_map3, patience_cnt = 0., 0
CKPT = MODEL_DIR / 'roberta_best.pt'

print(f'{"Epoch":>5} {"TrLoss":>8} {"ValLoss":>8} {"ValMAP3":>8}')
print('-'*36)
for ep in range(1, CFG['epochs']+1):
    tr_loss = train_epoch()
    va_loss, va_map3 = evaluate(va_loader)
    marker = ' *' if va_map3 > best_map3 else ''
    print(f'{ep:>5} {tr_loss:>8.4f} {va_loss:>8.4f} {va_map3:>8.4f}{marker}')

    # Log to WandB
    if USE_WANDB:
        wandb.log({
            'epoch'      : ep,
            'train/loss' : tr_loss,
            'val/loss'   : va_loss,
            'val/map3'   : va_map3,
        })

    # Early stopping + checkpoint
    if va_map3 > best_map3:
        best_map3, patience_cnt = va_map3, 0
        torch.save(model.state_dict(), CKPT)
    else:
        patience_cnt += 1
        if patience_cnt >= CFG['patience']:
            print(f'Early stop at epoch {ep}. Best val MAP@3 = {best_map3:.4f}')
            break

print(f'\nTraining done. Best val MAP@3 = {best_map3:.4f}')
if USE_WANDB:
    wandb.log({'best_val_map3': best_map3})

Epoch   TrLoss  ValLoss  ValMAP3
------------------------------------


/tmp/ipykernel_24/1696257451.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):


    1   1.3960   0.6400   0.8890 *
    2   0.5017   0.2005   0.9855 *
    3   0.2035   0.0681   0.9963 *
    4   0.1041   0.0298   0.9975 *
    5   0.0549   0.0197   0.9988 *

Training done. Best val MAP@3 = 0.9988


## 10. Inference + submission

Loads the best checkpoint, runs inference on the real test set, and saves
`submission_roberta_mc.csv`.

For each test question we take the top-3 options by softmax probability
and join them with spaces (e.g. `'A C D'`).

In [21]:
# Load best checkpoint
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()
print('Best checkpoint loaded.')

Best checkpoint loaded.


In [22]:
# Run inference on test set
all_logits = []
with torch.no_grad():
    for batch in te_loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        out  = model(input_ids=ids, attention_mask=mask)
        all_logits.append(out.logits.cpu())
all_logits = torch.cat(all_logits)
probs = torch.softmax(all_logits, dim=1).numpy()

In [23]:
# Build top-3 predictions per question
predictions = []
for i in range(len(test_df)):
    top3 = np.argsort(probs[i])[-3:][::-1]
    predictions.append(' '.join(REVERSE_MAP[j] for j in top3))

submission = pd.DataFrame({'ID': test_df['id'].values, 'Prediction': predictions})

In [24]:
# Format check: each prediction must be exactly 3 letters from A-E
errs = sum(1 for p in predictions if len(p.split())!=3 or not all(c in 'ABCDE' for c in p.split()))
print(f'Format errors: {errs} (must be 0)')

sub_path = PRED_DIR / 'submission_roberta_mc.csv'
submission.to_csv(sub_path, index=False)
print(f'Saved: {sub_path}')
submission.head(10)

Format errors: 0 (must be 0)
Saved: /kaggle/working/predictions/submission_roberta_mc.csv


,ID,Prediction
0,1,A D B
1,2,B E D
2,3,B D A
3,4,E C D
4,5,C A D
5,6,D B A
6,7,E D A
7,8,B E A
8,9,C D E
9,10,B D C


In [25]:
# close WandB run
if USE_WANDB:
    wandb.finish()
    print('Main training WandB run finished.')

wandb: uploading history steps 4-5, summary, console lines 11-14; updating run metadata
wandb: uploading history steps 4-5, summary, console lines 11-14; uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 4-5, summary, console lines 11-14
wandb: 
wandb: Run history:
wandb: best_val_map3 ▁
wandb:         epoch ▁▃▅▆█
wandb:    train/loss █▃▂▁▁
wandb:      val/loss █▃▂▁▁
wandb:      val/map3 ▁▇███
wandb: 
wandb: Run summary:
wandb: best_val_map3 0.99875
wandb:         epoch 5
wandb:    train/loss 0.05485
wandb:      val/loss 0.01966
wandb:      val/map3 0.99875
wandb: 
wandb: 🚀 View run roberta_mc_v1 at: https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026/runs/7awxxarm
wandb: ⭐️ View project at: https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260714_155558-7awxxarm/logs


Main training WandB run finished.


**Key architectural contrast:**
- **DeBERTa :** 5 independent binary
  (prompt, option) forward passes per question; scores compared afterwards.
  More training signal (5x pairs from the same data), needs manual handling
  of class imbalance (`pos_class_weight`), and needs a custom head.
- **RoBERTa (this notebook):** all 5 options seen together in one joint
  softmax; simpler code (no manual loss weighting, no custom head), but
  each training example is "used once" rather than expanded into 5, and
  batches are 5x wider in memory for the same number of questions.
